# DIY Copilot
## Lesson: Model Context Protocol (MCP)

In the last lesson, our tool was a Python function **inside** our harness: we wrote its description, and our tool loop executed it. That works – but only for our harness. If you want to use the same knowledge base search in GitHub Copilot, Claude Code, Cursor or your colleague's assistant, each of these harnesses needs its own integration.

The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) solves this. It is an open standard for connecting AI applications – like our harness – to tools and data:

```
 ┌──────────────────────────┐            ┌────────────────────────────────┐
 │ Harness (MCP host)       │            │ MCP server                     │
 │ e.g. our notebook,       │  MCP       │ e.g. company knowledge base    │
 │ Claude Code, VS Code     │ ◄────────► │                                │
 │   └─ MCP client          │ (JSON-RPC) │  tool: search_knowledge_base   │
 └──────────────────────────┘            └────────────────────────────────┘
```

* An **MCP server** offers tools (and optionally resources and prompts). It describes them itself.
* An **MCP client** – part of the harness, which the MCP specification calls the *host* – connects to servers, **discovers** their tools at runtime and calls them.
* Client and server talk **JSON-RPC**, either via stdin/stdout of a local process (`stdio`) or via HTTP for remote servers.

In this lesson we move **exactly the same function** from the last lesson – `search_knowledge_base(query)` – into an MCP server and connect our assistant to it.

During this lesson you will learn:
* how an MCP server is built and how it describes its tools
* what client and server actually exchange (under the hood)
* how to connect to a server, discover its tools and call them with the MCP SDK
* how to plug MCP tools into the tool loop from the last lesson
* how to write your own MCP server and use several servers at once
* how other harnesses (Claude Code, VS Code, ...) can use the same server

### Set up the environment

In [ ]:
# >>> COLAB SETUP <<<
# Placeholder: clone the workshop repository into the Colab workspace and make `lib` importable.
# When running locally, the following line is sufficient.
import sys
sys.path.append('../../../')

In [ ]:
import json
import os
import subprocess
import tempfile
from contextlib import AsyncExitStack, asynccontextmanager
from pathlib import Path

from litellm import completion
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.types import LATEST_PROTOCOL_VERSION

from lib.constants import PROJECT_ROOT, TICKETS_PATH
from lib.environment import load_environment
from lib.retry import call_with_retry

default_models = load_environment()
generation_model = default_models["generation_model"]
print(f"Generation model: {generation_model}")

SYSTEM_PROMPT = (
    "You are a helpful coding assistant for the developers of our company. "
    "Use the available tools when a question depends on internal rules or conventions of the company. "
    "If a question has several parts, search separately for each part. "
    "Base your answer only on the search results and cite the sources."
)

# The MCP server is a separate Python script next to this notebook.
# We build the path from the repository root, because the working directory differs (e.g. in Colab).
SERVER_SCRIPT = PROJECT_ROOT / "day-02" / "solutions" / "02_mcp" / "knowledge_base_mcp_server.py"

# The server writes its log (stderr) into this file
SERVER_LOG = Path(tempfile.gettempdir()) / "knowledge_base_mcp_server.log"

### The MCP server
Let's look at the server. It uses `FastMCP` from the official [MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk). Compare it with the last lesson:
* The function `search_knowledge_base` is the same – it only got the decorator `@mcp.tool()`.
* There is **no hand-written JSON schema** like in the last lesson. FastMCP generates it from the **type hints** and the **docstring**.
* The server builds the knowledge base index on first use and stores it on disk, so it is not rebuilt every time the server starts. Like in the last lesson, outdated documents (`archive/`) are not indexed.
* Logs go to **stderr**: with the `stdio` transport, stdout is the protocol channel. A single `print()` to stdout would break the communication.

In [ ]:
print(SERVER_SCRIPT.read_text(encoding="utf-8"))

### Under the hood: talking to the server by hand
MCP over `stdio` is surprisingly simple: the client starts the server as a subprocess and both exchange **one JSON-RPC message per line** via stdin and stdout. Before using the SDK, let's do it by hand to see what is exchanged.

The conversation always starts with a handshake:
1. `initialize` – the client tells the server which protocol version and capabilities it supports; the server answers with its name and capabilities.
2. `notifications/initialized` – the client confirms. (Notifications have no `id` and get no answer.)

After that, the client can list and call tools.

>**Note:** The server process needs the API key for the embedding model. A subprocess only inherits environment variables we pass to it, so we pass the current environment.

In [ ]:
server_log = open(SERVER_LOG, "w+")
server_process = subprocess.Popen(
    [sys.executable, str(SERVER_SCRIPT)],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=server_log,
    text=True,
    env=dict(os.environ),
)


def send(message: dict) -> None:
    """Sends one JSON-RPC message to the server (one line on stdin)."""
    print(f">>> {json.dumps(message)}")
    server_process.stdin.write(json.dumps(message) + "\n")
    server_process.stdin.flush()


def receive() -> dict:
    """Reads one JSON-RPC message from the server (one line on stdout)."""
    message = json.loads(server_process.stdout.readline())
    print(f"<<< {json.dumps(message)[:800]}\n")
    return message

In [ ]:
# 1. Handshake
send({
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": LATEST_PROTOCOL_VERSION,
        "capabilities": {},
        "clientInfo": {"name": "diy-copilot-notebook", "version": "1.0"},
    },
})
receive()
send({"jsonrpc": "2.0", "method": "notifications/initialized"})

In [ ]:
# TODO: 2. Ask the server which tools it offers (method "tools/list")
send({"jsonrpc": "2.0", "id": 2, "method": "tools/list"})
tools_list = receive()
print(json.dumps(tools_list["result"]["tools"][0], indent=1))

In [ ]:
# TODO: 3. Call the tool (method "tools/call" with the tool name and the arguments)
send({
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/call",
    "params": {"name": "search_knowledge_base", "arguments": {"query": "maximum duration of a meeting room booking in Berlin"}},
})
tool_result = receive()

In [ ]:
# Stop the server and have a look at its log (stderr)
server_process.terminate()
server_process.wait()
server_log.seek(0)
print(server_log.read()[-1500:])

Compare the result of `tools/list` with the tool declaration you wrote by hand in the last lesson: it's the same information – name, description and a JSON schema of the parameters (`inputSchema`). The difference: the **server** provides it, and the client **discovers** it at runtime.

### Exercise 01: Connect with the MCP SDK
Of course nobody writes JSON-RPC by hand. The MCP SDK provides a `ClientSession`, which handles the handshake, the request ids and the parsing for us.

The SDK uses Python's `async`/`await`. In a notebook you can use `await` directly in a cell. We write a small helper that starts the server, opens a session and closes everything again at the end of an `async with` block.

Your task is to complete the helper and use it to print the server's name and its tools.

In [ ]:
server_parameters = StdioServerParameters(
    command=sys.executable,
    args=[str(SERVER_SCRIPT)],
    env=dict(os.environ),  # pass the API key and the model defaults to the server process
)


# TODO: Complete the helper which starts the server and opens a session
@asynccontextmanager
async def connect_to_server():
    """Starts the MCP server and yields an initialized client session."""
    # errlog: the server's stderr goes into a log file instead of the notebook output
    with open(SERVER_LOG, "a") as errlog:
        async with stdio_client(server_parameters, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session

In [ ]:
# TODO: Connect to the server and list its tools
async with connect_to_server() as session:
    tools_result = await session.list_tools()

for tool in tools_result.tools:
    print(f"Tool: {tool.name}")
    print(f"Description: {tool.description[:200]}...")
    print(f"Input schema: {json.dumps(tool.inputSchema)}")

### Exercise 02: Call the tool via MCP
Your task is to call `search_knowledge_base` via the session and print the result. An MCP tool result consists of a list of **content blocks** (text, images, ...) and a flag `isError`.

In [ ]:
# TODO: Call the tool via the session
async with connect_to_server() as session:
    result = await session.call_tool("search_knowledge_base", {"query": "Which header do internal APIs use for API keys?"})

print(f"isError: {result.isError}")
print(result.content[0].text[:800])

### Exercise 03: MCP tools in the tool loop
Now we bring everything together. Our tool loop from the last lesson expects tool declarations in the OpenAI format and a way to execute a tool call. With MCP:
* the **declarations** come from `list_tools()` – we only have to convert them into the OpenAI format
* the **execution** is `session.call_tool()` – the function runs in the server process

Your task is to implement the conversion and the loop. Compare it with `run_assistant` from the last lesson: only two lines are really different.

In [ ]:
# TODO: Convert an MCP tool into an OpenAI-style tool declaration
def mcp_tool_to_openai(tool) -> dict:
    """Converts a tool from MCP's list_tools() into the OpenAI tool format used by LiteLLM."""
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description,
            "parameters": tool.inputSchema,
        },
    }


# TODO: Implement the tool loop with MCP tools
async def run_mcp_assistant(question: str, session: ClientSession, max_steps: int = 5) -> str:
    """Answers a question using the tools of an MCP server."""
    # Difference 1: discover the tools at runtime instead of defining them in our code
    tool_declarations = [mcp_tool_to_openai(tool) for tool in (await session.list_tools()).tools]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]

    for step in range(1, max_steps + 1):
        response = call_with_retry(completion, model=generation_model, messages=messages, tools=tool_declarations, temperature=0.0)
        message = response.choices[0].message
        messages.append(message.model_dump())

        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            arguments = json.loads(tool_call.function.arguments)
            print(f"[step {step}] {tool_call.function.name}({arguments})  -> via MCP")
            # Difference 2: the MCP server executes the function
            result = await session.call_tool(tool_call.function.name, arguments)
            content = "\n".join(block.text for block in result.content if block.type == "text")
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": content})

    return "I could not finish the task within the allowed number of steps."

In [ ]:
async with connect_to_server() as session:
    answer = await run_mcp_assistant(
        "A developer wants to cancel a meeting room booking in the Berlin office 1 hour before it starts via our API. "
        "Is that allowed, and which HTTP status code should the API return if it is not?",
        session,
    )
print(f"\n{answer}")

Our assistant now uses a tool it doesn't contain. If the knowledge base team adds a new tool to the server (e.g. `get_document(path)`), our assistant can use it **without any change** – it discovers it at the next `list_tools()`.

### Exercise 04: Write your own MCP server
Now it's your turn: connect your own data. Team Workplace has an export of its issue tracker (you know the tickets from the prompting lesson on day 1). Let's make it available to every assistant with a small MCP server offering two tools:
* `search_tickets(query, status)` – finds tickets whose title or description contains the query, optionally filtered by status
* `get_ticket(ticket_id)` – returns all details of one ticket

We write the server directly from the notebook into a file using the cell magic `%%writefile`. The path of the ticket export is passed as a command line argument, so the server doesn't depend on the workshop code.

Your task is to implement the two tools. Remember: FastMCP builds the tool description from the type hints and the docstring – write them carefully.

In [ ]:
TICKETS_SERVER_SCRIPT = Path(tempfile.gettempdir()) / "tickets_mcp_server.py"
print(TICKETS_SERVER_SCRIPT)

In [ ]:
%%writefile {TICKETS_SERVER_SCRIPT}
"""MCP server for the issue tracker export of Team Workplace."""

import json
import sys
from pathlib import Path

from mcp.server.fastmcp import FastMCP

TICKETS = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))

mcp = FastMCP("team-workplace-tickets", log_level="WARNING")


# TODO: Implement the tool search_tickets
@mcp.tool()
def search_tickets(query: str = "", status: str | None = None) -> str:
    """Searches the tickets of Team Workplace (the team that owns the Roomie service).

    Args:
        query: Text to search for in the title and the description, e.g. an error code or a component.
        status: Optional status filter: "open", "in progress", "done" or "rejected".

    Returns:
        A JSON list of matching tickets with id, title and status.
    """
    words = query.lower().split()
    results = [
        {"id": t["id"], "title": t["title"], "status": t["status"]}
        for t in TICKETS
        if all(word in (t["title"] + " " + t["description"]).lower() for word in words)
        and (status is None or t["status"] == status)
    ]
    return json.dumps(results)


# TODO: Implement the tool get_ticket
@mcp.tool()
def get_ticket(ticket_id: str) -> str:
    """Returns all details of a ticket, including description, workaround and resolution.

    Args:
        ticket_id: The id of the ticket, e.g. "WORK-412".
    """
    for ticket in TICKETS:
        if ticket["id"] == ticket_id:
            return json.dumps(ticket)
    return f"Error: ticket {ticket_id} does not exist."


if __name__ == "__main__":
    mcp.run()

In [ ]:
tickets_server_parameters = StdioServerParameters(
    command=sys.executable,
    args=[str(TICKETS_SERVER_SCRIPT), str(TICKETS_PATH)],
)

# Test the new server: list its tools and call one
@asynccontextmanager
async def connect(parameters: StdioServerParameters):
    """Starts an MCP server and yields an initialized client session."""
    with open(SERVER_LOG, "a") as errlog:
        async with stdio_client(parameters, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session


async with connect(tickets_server_parameters) as session:
    print([tool.name for tool in (await session.list_tools()).tools])
    result = await session.call_tool("search_tickets", {"query": "500", "status": "open"})
print(result.content[0].text)

**Several servers at once.** Real harnesses like Claude Code are connected to many MCP servers at the same time – and the model sees the tools of all of them. The harness has to remember which tool belongs to which server, so that it can send each call to the right one.

Your task is to complete `connect_to_servers` and the tool loop `run_multi_server_assistant`, which uses the knowledge base server **and** your ticket server.

>**Note:** Tool names must be unique across all servers. Harnesses often prefix them with the server name (e.g. `tickets__search_tickets`) to avoid collisions.

In [ ]:
# TODO: Connect to several servers at once and keep all sessions open
@asynccontextmanager
async def connect_to_servers(parameters_by_name: dict[str, StdioServerParameters]):
    """Starts several MCP servers and yields a dict: server name -> client session."""
    async with AsyncExitStack() as stack:
        sessions = {}
        for name, parameters in parameters_by_name.items():
            sessions[name] = await stack.enter_async_context(connect(parameters))
        yield sessions


# TODO: Implement the tool loop with the tools of all servers
async def run_multi_server_assistant(question: str, sessions: dict[str, ClientSession], max_steps: int = 6) -> str:
    """Answers a question using the tools of several MCP servers."""
    # Collect the tools of all servers and remember which session provides which tool
    tool_declarations, session_for_tool = [], {}
    for name, session in sessions.items():
        for tool in (await session.list_tools()).tools:
            tool_declarations.append(mcp_tool_to_openai(tool))
            session_for_tool[tool.name] = session

    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        response = call_with_retry(completion, model=generation_model, messages=messages, tools=tool_declarations, temperature=0.0)
        message = response.choices[0].message
        messages.append(message.model_dump())
        if not message.tool_calls:
            return message.content
        for tool_call in message.tool_calls:
            arguments = json.loads(tool_call.function.arguments)
            print(f"[step {step}] {tool_call.function.name}({arguments})")
            # Send the call to the server that provides the tool
            result = await session_for_tool[tool_call.function.name].call_tool(tool_call.function.name, arguments)
            content = "\n".join(block.text for block in result.content if block.type == "text")
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": content})
    return "I could not finish the task within the allowed number of steps."

In [ ]:
# A question that needs both servers: the ticket tracker and the knowledge base
async with connect_to_servers({"knowledge-base": server_parameters, "tickets": tickets_server_parameters}) as sessions:
    answer = await run_multi_server_assistant(
        "Developers report a 500 error when they create a booking from the mobile app. "
        "Is this a known problem, and which status code should the API return instead according to our guidelines?",
        sessions,
    )
print(f"\n{answer}")

### Outlook: Use the same server in other harnesses
The real value of MCP: **one server, many clients**. The same `knowledge_base_mcp_server.py` can be added to the AI tools your developers already use. Try one of them on your own machine (not in Colab):

**MCP Inspector** – the "Postman for MCP servers". It lists the tools and lets you call them by hand:
```bash
npx @modelcontextprotocol/inspector python day-02/solutions/02_mcp/knowledge_base_mcp_server.py
```

**Claude Code**:
```bash
claude mcp add company-knowledge-base -- python /path/to/knowledge_base_mcp_server.py
```

**VS Code (GitHub Copilot agent mode)** – in `.vscode/mcp.json`:
```json
{
  "inputs": [
    {"type": "promptString", "id": "gemini-api-key", "description": "Gemini API key", "password": true}
  ],
  "servers": {
    "company-knowledge-base": {
      "type": "stdio",
      "command": "python",
      "args": ["/path/to/knowledge_base_mcp_server.py"],
      "env": {"GEMINI_API_KEY": "${input:gemini-api-key}"}
    }
  }
}
```

Many companies offer hosted MCP servers for their products (e.g. Atlassian for Jira and Confluence, GitHub). They are connected via HTTP instead of `stdio` – the client code stays the same.

### Security considerations
MCP makes it easy to connect tools – and just as easy to connect the wrong ones. Remember the guardrails lesson:
* **A local MCP server is a program running on your machine** with your permissions. Only install servers you trust.
* **Tool descriptions and tool results go straight into the prompt.** A malicious server can hide instructions in them ("tool poisoning") – the same problem as indirect prompt injection.
* **Remote servers see your data.** Check what is sent to them, especially personal data and secrets (AI-02).
* Tools with side effects need the same checks as before: validation, output guards and human approval.

### Learnings
* MCP standardizes how harnesses **discover** and **call** tools: a server describes its tools, any client can use them.
* Under the hood, MCP is JSON-RPC: `initialize`, `tools/list`, `tools/call` – via stdin/stdout for local servers or HTTP for remote servers.
* The MCP SDK generates the tool schema from type hints and docstrings – no hand-written JSON.
* The tool loop doesn't change: the declarations come from `list_tools()`, the execution is `call_tool()`.
* Writing your own MCP server takes only a few lines – that's how you connect your own data and services to any assistant. With several servers, the harness routes each tool call to the right server.
* With stdio servers, stdout belongs to the protocol – log to stderr.
* Treat MCP servers like any other dependency: trust, review, and guard what goes in and out.